# 04_gold_regional_operations_dashboard

## Purpose

Create the Gold regional operations dashboard table.

This notebook combines Gold market, weather, and grid incident outputs into one dashboard-ready dataset.

## Expected output

`vattenfall_dev.analytics.gold_regional_operations_dashboard`

## Expected grain

One row per report day and region.

## Expected KPIs

- market price context
- market volume context
- weather condition context
- grid incident counts
- elevated event counts
- operational attention flag

## Main idea

This table is designed as the main dashboard-ready Gold output for regional energy operations and market intelligence.

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from pyspark.sql import functions as F
from utils.logging_utils import log_table_operation, log_row_count

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from pyspark.sql import functions as F
from utils.logging_utils import log_table_operation, log_row_count

In [0]:
catalog = "vattenfall_dev"

market_gold_table = f"{catalog}.analytics.gold_daily_market_summary"
weather_gold_table = f"{catalog}.analytics.gold_weather_impact_summary"
grid_gold_table = f"{catalog}.analytics.gold_grid_incident_summary"

target_table = f"{catalog}.analytics.gold_regional_operations_dashboard"

print("Gold table:", "Regional Operations Dashboard")
print("Market source:", market_gold_table)
print("Weather source:", weather_gold_table)
print("Grid source:", grid_gold_table)
print("Target table:", target_table)
print("Grain: one row per report_day and region")

In [0]:
market_df = spark.table(market_gold_table)
weather_df = spark.table(weather_gold_table)
grid_df = spark.table(grid_gold_table)

log_row_count("Gold market summary rows", market_df.count())
log_row_count("Gold weather impact rows", weather_df.count())
log_row_count("Gold grid incident rows", grid_df.count())

display(market_df)
display(weather_df)
display(grid_df)

In [0]:
grid_day_region_df = (
    grid_df
    .groupBy(
        F.col("event_day").alias("report_day"),
        "region"
    )
    .agg(
        F.sum("incident_count").alias("total_incident_count"),
        F.sum("elevated_incident_count").alias("total_elevated_incident_count"),
        F.sum("total_duration_minutes").alias("total_incident_duration_minutes"),
        F.round(F.avg("avg_duration_minutes"), 2).alias("avg_incident_duration_minutes"),
        F.sum("affected_asset_count").alias("total_affected_asset_count"),
    )
)

display(grid_day_region_df)

In [0]:
dashboard_df = (
    market_df.alias("m")
    .join(
        weather_df.alias("w"),
        on=["report_day", "region"],
        how="left"
    )
    .join(
        grid_day_region_df.alias("g"),
        on=["report_day", "region"],
        how="left"
    )
    .fillna(
        {
            "total_incident_count": 0,
            "total_elevated_incident_count": 0,
            "total_incident_duration_minutes": 0,
            "avg_incident_duration_minutes": 0,
            "total_affected_asset_count": 0,
        }
    )
    .withColumn(
        "operational_attention_flag",
        F.when(
            (F.col("is_high_market_price") == 1)
            | (F.col("weather_risk_signal") == "WEATHER_RISK")
            | (F.col("total_elevated_incident_count") > 0),
            "ATTENTION"
        ).otherwise("NORMAL")
    )
)

dashboard_count = dashboard_df.count()

log_row_count("Gold regional operations dashboard rows", dashboard_count)

display(dashboard_df)

In [0]:
(
    dashboard_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

log_table_operation(
    source_table="Gold market, weather, and grid summaries",
    target_table=target_table,
    operation_name="Create Gold regional operations dashboard"
)

print("Written Gold dashboard table:", target_table)

In [0]:
gold_df = spark.table(target_table)

print("Rows in dashboard Gold table:", gold_df.count())
print("Columns:", gold_df.columns)

display(gold_df)

In [0]:
print("Operational attention flag distribution:")

gold_df.groupBy("operational_attention_flag").count().show()

print("Null checks for dashboard grain:")

for column_name in ["report_day", "region"]:
    null_count = gold_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Duplicate grain check:")

duplicate_count = (
    gold_df
    .groupBy("report_day", "region")
    .count()
    .filter("count > 1")
    .count()
)

print("duplicate day-region rows:", duplicate_count)